# Best-of-N — đúng SD1.5 stack TPU đang dùng

Baseline: N root độc lập, mỗi root full100 DDIM, chỉ chấm **ảnh terminal** bằng ImageReward, chọn argmax. Mặc định **N24×100=2400NFE**, không scout/correction/resampling. Giữ BF16, DDIM stochastic η1, CFG7.5,512px và original per-step noise stream.

**Chỉ dùng trên VM hiện có** với kernel `/home/thanhlamtba31/static-bfs-dcs/venv/bin/python`. Không pip install/upgrade sampler environment. Chạy notebook một lần mỗi kernel để không khởi tạo lại XLA cache/device. Các cell model chưa được chạy sẵn; không có số đo hoặc ảnh giả.

Nếu chỉ cần GPU/Colab độc lập, dùng `best_of_n_sd15_gpu.ipynb`.

## 1. Kiểm tra môi trường và chip trước khi import XLA

In [ ]:
import os,sys,subprocess
from pathlib import Path
# Run this cell BEFORE importing torch_xla or initializing any TPU object.
os.environ['PJRT_DEVICE']='TPU'
os.environ['TPU_VISIBLE_CHIPS']='0'  # Use one free chip; inspect owners below.
os.environ['HF_HUB_OFFLINE']='1';os.environ['TRANSFORMERS_OFFLINE']='1'
os.environ['OMP_NUM_THREADS']='4';os.environ['MKL_NUM_THREADS']='4';os.environ['SINKHORN_CPU_THREADS']='4'
LEGACY_REPO=Path('/home/thanhlamtba31/TTS-reserach')
DCS_ROOT=Path('/home/thanhlamtba31/static-bfs-dcs')
ADAPTER_CONFIG=DCS_ROOT/'ot_tts_dinov2_pilot_20261016/attempts/8074f94c847d/adapter_config.json'
assert LEGACY_REPO.exists() and ADAPTER_CONFIG.exists(),'This notebook requires the existing TPU VM stack.'
assert Path(sys.prefix).resolve()==(DCS_ROOT/'venv').resolve(),f'Select the existing TPU venv kernel; current Python: {sys.executable}'
owners=subprocess.run(['fuser','/dev/vfio/0'],capture_output=True,text=True)
assert not owners.stdout.strip(),f'Chip0 is owned; inspect the process before continuing: {owners.stdout}'
sys.path.insert(0,str(LEGACY_REPO))
print('Python:',sys.executable,'; chip0 has no current fuser owner')

## 2. Helpers và tham số

In [ ]:
import csv,hashlib,importlib.metadata,json,math,time,uuid
from pathlib import Path
import numpy as np
import torch
from PIL import Image,ImageDraw
from IPython.display import display

def versions():
    result={}
    for package in ['torch','torchvision','diffusers','transformers','image-reward','numpy','Pillow','torch_xla']:
        try:result[package]=importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError:pass
    return result

class UNetMeter:
    """Count actual samples through UNet, separately from batched forward calls."""
    def __init__(self,cfg_branches=2):
        self.branches=cfg_branches;self.sample_nfe=0;self.cfg_sample_evaluations=0;self.forward_calls=0
    def hook(self,module,args,kwargs):
        x=args[0] if args else kwargs['sample']
        count=int(x.shape[0]);assert count%self.branches==0
        self.sample_nfe+=count//self.branches
        self.cfg_sample_evaluations+=count;self.forward_calls+=1
    def as_dict(self):
        return dict(sample_nfe=self.sample_nfe,cfg_branch_sample_evaluations=self.cfg_sample_evaluations,
                    actual_batched_unet_forward_calls=self.forward_calls)

def write_result(prompt,images,scores,metadata,candidate_keys,out_root):
    scores=np.asarray(scores,dtype=np.float64).reshape(-1)
    assert len(images)==len(scores)==len(candidate_keys)>0 and np.isfinite(scores).all()
    # Stable tie-break: candidate with smaller index wins equal ImageReward.
    order=sorted(range(len(scores)),key=lambda i:(-float(scores[i]),i));best=order[0]
    folder=Path(out_root)/('run_'+time.strftime('%Y%m%d_%H%M%S')+'_'+uuid.uuid4().hex[:8]);folder.mkdir(parents=True,exist_ok=False)
    rows=[]
    for i,image in enumerate(images):
        path=folder/f'candidate_{i:03d}.png';image.save(path)
        rows.append(dict(candidate_id=i,candidate_key=str(candidate_keys[i]),image_reward=float(scores[i]),
                         image=path.name,image_sha256=hashlib.sha256(path.read_bytes()).hexdigest()))
    with (folder/'scores.csv').open('w',newline='') as f:
        w=csv.DictWriter(f,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)
    images[best].save(folder/'best.png')
    cols=min(6,len(images));side=160;lines=math.ceil(len(images)/cols)
    sheet=Image.new('RGB',(cols*side,lines*(side+27)),'white');draw=ImageDraw.Draw(sheet)
    for pos,i in enumerate(order):
        x=(pos%cols)*side;y=(pos//cols)*(side+27)
        sheet.paste(images[i].resize((side,side)),(x,y))
        draw.text((x+3,y+side+3),f'#{i} IR={float(scores[i]):.4f}',fill='black')
    sheet.save(folder/'ranked_grid.jpg',quality=92)
    summary=dict(method='best_of_n',prompt=prompt,best_candidate_id=best,best_image_reward=float(scores[best]),
                 mean_image_reward=float(scores.mean()),ranking=order,candidates=rows,**metadata)
    (folder/'summary.json').write_text(json.dumps(summary,indent=2,ensure_ascii=False,default=str)+'\n')
    print(f'Best candidate #{best}, ImageReward={scores[best]:.6f}; output: {folder}')
    print(json.dumps(metadata['accounting'],indent=2));display(images[best]);display(sheet)
    return summary,folder

In [ ]:
PROMPT = 'A red canoe floating on a still alpine lake with yellow autumn trees and a distant snow-covered mountain.'
PROMPT_ID = 1  # Must match the paired experiment, not just the prompt text.
N = 24
BASE_SEED = 2026102101
OUT_ROOT = Path('/dev/shm/bon_notebook_results')
assert isinstance(N,int) and N>0 and N%4==0,'Frozen sampler microbatch4: require N divisible by4, no hidden padding.'
config=json.loads(ADAPTER_CONFIG.read_text())
for file,expected in config['legacy_source_hashes'].items():
    assert hashlib.sha256(Path(file).read_bytes()).hexdigest()==expected,f'Legacy source drift: {file}'
SAMPLER_CONFIG=config['sampler']
assert SAMPLER_CONFIG['steps']==100 and SAMPLER_CONFIG['ddim_eta']==1 and SAMPLER_CONFIG['cfg_scale']==7.5
assert SAMPLER_CONFIG['batch_size']==4 and SAMPLER_CONFIG['resolution']==512
assert SAMPLER_CONFIG['revision']=='451f4fe16113bff5a5d2269ed5ad43b0592e9a14'
print(f'BoN{N}: {N*100} sample NFE; {N*200} CFG-branch sample evaluations.')

## 3. Load sampler/verifier cũ, xác minh source hashes

In [ ]:
from sd15_sinkhorn_tpu.tpu_runtime import Runtime
from sd15_sinkhorn_tpu.sd15_sampler import Sampler
from sd15_sinkhorn_tpu.reward import ImageRewardCPU
from sd15_sinkhorn_tpu.common import initial_noise
runtime_dir=OUT_ROOT/('runtime_'+uuid.uuid4().hex[:8]);runtime_dir.mkdir(parents=True,exist_ok=False)
start=time.perf_counter()
rt=Runtime(str(runtime_dir),rank=0)
sampler=Sampler(rt,SAMPLER_CONFIG,str(DCS_ROOT))
reward_model=ImageRewardCPU(str(DCS_ROOT))
model_load_seconds=time.perf_counter()-start
print('Frozen BF16 sampler loaded. No full100 warmup or preview UNet calls were added.')

## 4. Full100 → ImageReward → lấy tốt nhất

N phải chia hết cho4 để không có padding. Không chạy thêm warmup, scout, auxiliary clean prediction hoặc bank. Seed/prompt ID/root ordering và microbatch4 giữ nguyên để dùng chung initial pool và future noise với baseline BFS trên stack này.

ImageReward giữ một batchN cố định: đổi verifier batch shape có thể lệch rất nhỏ về số học. Khi cần replay score bitwise, giữ đúng ảnh, thứ tự và kích thước batch của đối chứng. Cùng integer seed trên GPU và TPU không bảo đảm cùng trajectory.

In [ ]:
initial,noise_seed=initial_noise(BASE_SEED,PROMPT_ID,0,N)
meter=UNetMeter(cfg_branches=2)
handle=sampler.pipe.unet.register_forward_pre_hook(meter.hook,with_kwargs=True)
rt.sync();start=time.perf_counter()
try:
    final,_,_=sampler.run(PROMPT,initial,start=0,checkpoints=(),noise_seed=noise_seed)
    rt.sync()
finally:
    handle.remove()
generation_seconds=time.perf_counter()-start
assert final.shape==(N,4,64,64) and torch.isfinite(final).all()
assert meter.sample_nfe==N*100 and meter.cfg_sample_evaluations==N*200
# Source-keyed historical noise stream from the original Sampler.run, unchanged.
# Base key: SeedSequence([BASE_SEED,PROMPT_ID]); per-step generator:
# noise_seed + index*100003 + microbatch_offset. Same N/batch/IDs pair with BFS24.
start=time.perf_counter();images=sampler.decode(final);decode_seconds=time.perf_counter()-start
start=time.perf_counter();scores=reward_model.score(PROMPT,images);reward_seconds=time.perf_counter()-start
assert len(images)==len(scores)==N
accounting=dict(**meter.as_dict(),nominal_sample_nfe=N*100,physical_padding_sample_nfe=0,
                auxiliary_unet_nfe=0,scout_nfe=0,terminal_reward_scores=N,terminal_reward_calls=1,
                reward_model_batch_size=N,intermediate_reward_scores=0)
keys=[f'master_seed={BASE_SEED};prompt_id={PROMPT_ID};stream=0;root={i};noise_seed={noise_seed}' for i in range(N)]
summary,output_dir=write_result(PROMPT,images,scores,dict(
    backend='existing_sd15_tpu_frozen',model_id=SAMPLER_CONFIG['model'],model_revision=SAMPLER_CONFIG['revision'],
    n=N,steps=100,cfg=7.5,ddim_eta=1.,dtype='torch.bfloat16',batch_size=4,
    master_seed=BASE_SEED,prompt_id=PROMPT_ID,noise_seed=int(noise_seed),versions=versions(),
    scheduler=dict(sampler.scheduler.config),timesteps=sampler.timesteps,
    legacy_source_hashes=config['legacy_source_hashes'],runtime_info=rt.info,
    model_load_seconds=model_load_seconds,generation_seconds=generation_seconds,
    decode_seconds=decode_seconds,reward_seconds=reward_seconds,accounting=accounting,
    reward_api='existing ImageRewardCPU.score_batched, one fixed full-N batch'
),keys,OUT_ROOT)
torch.save(dict(initial=initial,final=final,master_seed=BASE_SEED,prompt_id=PROMPT_ID,
                noise_seed=int(noise_seed),rewards=torch.tensor(scores)),output_dir/'latents.pt')
print('RAM-disk output is temporary; download/archive output_dir before VM restart.')

## 5. Xuất kết quả

`output_dir` có tất cảPNG, winner `best.png`, grid,CSV,JSON và `latents.pt`. Output ởRAM disk để tránh đầy ổ VM; tải xuống hoặc lưu vào kho artifact trước khi tắt/restart VM. Không tự upload, xóa dữ liệu hay tắt VM trong notebook.

Chọn winner bằng **max**, không phải min. Không dùng intermediate x0 reward. TổngNFE làN×100; CFG branches=2×NFE. VAE/ImageReward được ghi riêng, không giấu thành denoising NFE. Đây là BoN24/2400, không được ghi thành BoN2360. IR chỉ đo objective dùng để selection, không phải kiểm chứng semantic success.

Đối chiếu source/runtime: `summary.json` lưu model revision, schedule, legacy source hashes, noise seed, prompt ID và package versions. Verifier giao diện gốc theo [ImageReward](https://github.com/zai-org/ImageReward); sampler là implementation gốc trên VM, không thay bằng pipeline GPU.